# 💡 FinSight — MSME Financial Document Intelligence

**Semantic RAG + NLI faithfulness + loan eligibility scoring**  
Zero paid APIs · Runs fully on CPU

---
**Demo flow (90 seconds):**
1. Upload a financial PDF (balance sheet / P&L / bank statement)
2. Ask: *"What is the net profit margin?"*
3. Show faithfulness score (NLI entailment probability)
4. Navigate to Insights → live loan eligibility gauge

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q \
    streamlit \
    pdfplumber \
    'sentence-transformers>=2.6.0' \
    faiss-cpu \
    'transformers>=4.35.0' \
    torch \
    plotly \
    scipy \
    pyngrok

print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Upload project files ───────────────────────────────────────────────
# Option A: Upload finsight.zip using the Files panel, then run:
import zipfile, os
with zipfile.ZipFile('finsight.zip', 'r') as z:
    z.extractall('.')
print('✅ Project extracted')

# Option B: If you pushed to GitHub, use:
# !git clone https://github.com/YOUR_USERNAME/finsight.git

In [ ]:
# ── Cell 3: Pre-download models (optional — avoids UI lag on first query) ──────
print('Downloading MiniLM embedder...')
from sentence_transformers import SentenceTransformer
SentenceTransformer('all-MiniLM-L6-v2')

print('Downloading Flan-T5...')
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
AutoTokenizer.from_pretrained('google/flan-t5-base')
AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')

print('Downloading NLI cross-encoder...')
from sentence_transformers.cross_encoder import CrossEncoder
CrossEncoder('cross-encoder/nli-MiniLM2-L6-H768')

print('✅ All models cached')

In [ ]:
# ── Cell 4: Launch FinSight ────────────────────────────────────────────────────
import subprocess, threading, time
from pyngrok import ngrok

# Kill any existing Streamlit or ngrok processes
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
ngrok.kill()
time.sleep(2)

# Start Streamlit in background thread
def _run():
    subprocess.run([
        'streamlit', 'run', 'finsight/app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
    ])

threading.Thread(target=_run, daemon=True).start()
time.sleep(4)

# Create public tunnel
public_url = ngrok.connect(8501)
print('\n' + '='*50)
print(f'💡 FinSight is live!')
print(f'🔗 URL: {public_url}')
print('='*50)